In [5]:
import Pkg

# Activate your MyProject environment
Pkg.activate(raw"C:\Users\HP\Code\Master thesis\Model\MyJulia19")

# Make sure all dependencies are installed according to Project.toml/Manifest.toml
# Pkg.instantiate()

# Confirm the active environment
println("Active environment: ", Base.active_project())

Active environment: C:\Users\HP\Code\Master thesis\Model\MyJulia19\Project.toml


  Activating project at `C:\Users\HP\Code\Master thesis\Model\MyJulia19`


In [6]:
using Random, Plots
using Zygote, ForwardDiff
using OrdinaryDiffEq, DiffEqSensitivity
using LinearAlgebra
using Statistics
using ProgressBars, Printf
using Flux
using Flux.Optimise: update!
using Flux.Losses: mae, mse
using BSON: @save, @load
using DelimitedFiles
using YAML
using MLFlowClient

In [7]:
pwd()

"C:\\Users\\HP\\Code\\Master thesis\\Model\\Xylan\\No catalyst"

In [8]:
ENV["GKSwstype"] = "100"

"100"

In [97]:
expr_name = "Xylan-5s5r-01"  # Experiment identifier
fig_path = string("./results_nocat/", expr_name, "/figs")  # Path to save figures
ckpt_path = string("./results_nocat/", expr_name, "/checkpoint")  # Path to save model checkpoints
is_restart = false  # Whether to restart from a checkpoint
ns = 5  #  number of species
nr = 5  #  number of reactions

lb = 1.e-8  # Likely a lower bound or tolerance value
n_epoch = 1500  # Total training epochs
n_plot = 10  # How often to generate plots during training
grad_max = 1.e2  # Gradient clipping threshold
maxiters = 10000  # Maximum optimization iterations

lr_max = 5.e-3  # Maximum learning rate
lr_min = 1.e-5  # Minimum learning rate
lr_decay = 0.2  # Learning rate decay factor
lr_decay_step = 375  # Steps between LR decays
w_decay = 1.e-8  # Weight decay (L2 regularization) coefficient

1.0e-8

In [98]:
# Create MLFlow instance
mlf = MLFlow("http://localhost:5000/api")

# Initiate new experiment

# experiment_id = createexperiment(mlf, "Xylan No Cat")

mlf_exp = getexperimentbyname(mlf, "Xylan No Cat")
experiment_id = mlf_exp.experiment_id

# Create a run in the new experiment

exprun = createrun(mlf, experiment_id)
# old_run_id = "11a6f4d80ca24fb4976f3f924913a1ca"  # from UI
# exprun = getrun(mlf, old_run_id)


Run(
    info = RunInfo(
    run_id = "17760639273a4df6a0869f503f792c2a", 
    run_name = "welcoming-worm-531", 
    experiment_id = "784704581357715807", 
    status = MLFlowClient.RunStatus.RUNNING, 
    start_time = 0, 
    end_time = nothing, 
    artifact_uri = "mlflow-artifacts:/784704581357715807/17760639273a4df6a0869f503f792c2a/artifacts", 
    lifecycle_stage = "active"
), 
    data = RunData(
    metrics = Metric[], 
    params = Param[], 
    tags = Tag[Tag(
    key = "mlflow.runName", 
    value = "welcoming-worm-531"
)]
), 
    inputs = RunInputs(
    dataset_inputs = DatasetInput[], 
    model_inputs = ModelInput[]
), 
    outputs = MLFlowClient.RunOutputs(
    model_outputs = ModelOutput[]
)
)

In [99]:
llb = lb;
global p_cutoff = -1.0

const l_exp = 1:10
n_exp = length(l_exp)

l_train = []
l_val = []
for i = 1:n_exp
    j = l_exp[i]
    if !(j in [3, 6, 9]) # These are used as validation
        push!(l_train, i)
    else
        push!(l_val, i)
    end
end

opt = Flux.Optimise.Optimiser(
  Flux.Optimise.ExpDecay(lr_max, lr_decay, length(l_train) * lr_decay_step, lr_min),
  Flux.Optimise.AdamW(lr_max, (0.9, 0.999), w_decay),  # Changed from ADAMW to AdamW
)

if !is_restart
    if ispath(fig_path)
        rm(fig_path, recursive = true)
    end
    if ispath(ckpt_path)
        rm(ckpt_path, recursive = true)
    end
end

if ispath("./results_nocat") == false
    mkdir("./results_nocat")
end

if ispath("./results_nocat/$expr_name") == false
    mkdir("./results_nocat/$expr_name")
end

if ispath(fig_path) == false
    mkdir(fig_path)
    mkdir(string(fig_path, "/conditions"))
end

if ispath(ckpt_path) == false
    mkdir(ckpt_path)
end

#cp("./config.yaml", config_path; force=true)

"./results_nocat/Xylan-5s5r-01/checkpoint"

In [100]:
l_train, l_val, n_exp

(Any[1, 2, 4, 5, 7, 8, 10], Any[3, 6, 9], 10)

In [101]:
function load_exp(filename)
    exp_data = readdlm(filename)
    index = indexin(unique(exp_data[:, 1]), exp_data[:, 1])
    exp_data = exp_data[index, :]
    exp_data[:, 3] = exp_data[:, 3] / maximum(exp_data[:, 3])

    # ✅ NEW: enforce monotone decrease — remove TGA artefact
    for i in 2:size(exp_data, 1)
        if exp_data[i, 3] > exp_data[i-1, 3]
            exp_data[i, 3] = exp_data[i-1, 3]
        end
    end
    return exp_data
end


load_exp (generic function with 1 method)

In [102]:
l_exp_data = [];
l_exp_info = zeros(Float64, length(l_exp), 2);
for (i_exp, value) in enumerate(l_exp)
    filename = string("data_no_cat/expdata_no", string(value), ".txt")

    exp_data = Float64.(load_exp(filename))

    push!(l_exp_data, exp_data)
    l_exp_info[i_exp, 1] = exp_data[1, 2] # initial temperature, K
end
l_exp_info[:, 2] = readdlm("data_no_cat/beta.txt")[l_exp];

In [103]:
l_exp_data[1]

182×3 Matrix{Float64}:
  2362.53  402.607  1.0
  2427.06  405.242  0.996875
  2491.59  407.834  0.99375
  2556.12  410.437  0.99375
  2620.65  412.997  0.990339
  2685.18  415.631  0.990339
  2749.71  418.236  0.990339
  2814.24  420.87   0.9875
  2878.77  423.513  0.9875
  2943.3   426.172  0.9875
  3007.83  428.878  0.984375
  3072.36  431.525  0.984375
  3136.89  434.237  0.98125
     ⋮              
 11203.2   770.487  0.0156246
 11267.7   773.182  0.0156246
 11332.2   775.849  0.0124934
 11396.8   778.55   0.0124934
 11461.3   781.257  0.009375
 11525.8   783.927  0.009375
 11590.3   786.597  0.00625
 11654.9   789.29   0.00625
 11719.4   791.945  0.003125
 11783.9   794.638  0.003125
 11848.5   797.313  7.03e-26
 11913.0   799.96   0.0

In [104]:
l_exp_info

10×2 Matrix{Float64}:
 402.607   2.5
 402.636   2.5
 402.621   2.5
 402.582   2.5
 403.644   5.0
 403.589   5.0
 403.639   5.0
 403.926  10.0
 403.829  10.0
 403.816  10.0

In [105]:
Random.seed!(42)
np = nr * (ns + 3) + 1
p = randn(Float64, np) .* 1.e-2;
p[1:nr] .+= 0.8;  # w_b
p[nr*(ns+1)+1:nr*(ns+2)] .+= 0.8;  # w_out
p[nr*(ns+2)+1:nr*(ns+3)] .+= 0.1;  # w_b | w_Ea
p[end] = 0.1;  # slope

In [106]:
function p2vec(p)
    slope = p[end] .* 1.e1
    w_b = p[1:nr] .* (slope * 10.0)
    w_b = clamp.(w_b, -23, 50)

    w_out = reshape(p[nr+1:nr*(ns+1)], ns, nr)
    @. w_out[1, :] = clamp(w_out[1, :], -3.0, 0.0)
    @. w_out[end, :] = clamp(abs(w_out[end, :]), 0.0, 3.0)

    if p_cutoff > 0.0
        w_out[findall(abs.(w_out) .< p_cutoff)] .= 0.0
    end

    w_out[ns-1:ns-1, :] .=
        -sum(w_out[1:ns-2, :], dims = 1) .- sum(w_out[ns:ns, :], dims = 1)

    w_in_Ea = abs.(p[nr*(ns+1)+1:nr*(ns+2)] .* (slope * 100.0))
    w_in_Ea = clamp.(w_in_Ea, 100.0, 300.0) #Constrain parameters changes according to the species. Ea

    w_in_b = abs.(p[nr*(ns+2)+1:nr*(ns+3)])

    #w_in_ocen = abs.(p[nr*(ns+3)+1:nr*(ns+4)])
    #w_in_ocen = clamp.(w_in_ocen, 0.0, 1.5)

    #if p_cutoff > 0.0
    #    w_in_ocen[findall(abs.(w_in_ocen) .< p_cutoff)] .= 0.0
    #end

    #w_in = vcat(clamp.(-w_out, 0.0, 4.0), w_in_Ea', w_in_b', w_in_ocen')
    w_in = vcat(clamp.(-w_out, 0.0, 4.0), w_in_Ea', w_in_b')
    return w_in, w_b, w_out
end

p2vec (generic function with 1 method)

In [107]:
function display_p(p)
    w_in, w_b, w_out = p2vec(p)
    println("\n species (column) reaction (row)")
    println("w_in | Ea | b | lnA | w_out")
    show(stdout, "text/plain", round.(hcat(w_in', w_b, w_out'), digits = 3))
    # println("\n w_out")
    # show(stdout, "text/plain", round.(w_out', digits=3))
    println("\n")
end
display_p(p)


 species (column) reaction (row)
w_in | Ea | b | lnA | w_out
5×13 Matrix{Float64}:
 -0.0    0.009  0.015  0.0    0.0  100.0  0.096  7.964   0.0    -0.009  -0.015   0.023  0.0
  0.008  0.0    0.0    0.01   0.0  100.0  0.106  8.025  -0.008   0.008   0.004  -0.01   0.005
 -0.0    0.001  0.013  0.0    0.0  100.0  0.099  7.969   0.0    -0.001  -0.013   0.013  0.001
  0.014  0.015  0.0    0.0    0.0  100.0  0.092  7.969  -0.014  -0.015   0.013   0.0    0.015
 -0.0    0.0    0.0    0.018  0.0  100.0  0.091  8.082   0.0     0.009   0.003  -0.018  0.007



In [108]:
function getsampletemp(t, T0, beta, t0 = 0.0)
    if beta < 100
        T = T0 + beta / 60 * (t - t0)  # K/min to K/s
    else
        tc = [999.0, 1059.0] .* 60.0
        Tc = [beta, 370.0, 500.0] .+ 273.0
        HR = 40.0 / 60.0
        if t <= tc[1]
            T = Tc[1]
        elseif t <= tc[2]
            T = minimum([Tc[1] + HR * (t - tc[1]), Tc[2]])
        else
            T = minimum([Tc[2] + HR * (t - tc[2]), Tc[3]])
        end
    end
    return T
end

getsampletemp (generic function with 2 methods)

In [109]:
const R = -1.0 / 8.314e-3  # universal gas constant, kJ/mol*K
@inbounds function crnn!(du, u, p, t)
    logX = @. log(clamp(u, lb, 10.0))
    T = getsampletemp(t, T0, beta, t0_exp)
    #w_in_x = w_in' * vcat(logX, R / T, log(T), ocen)
    w_in_x = w_in' * vcat(logX, R / T, log(T))
    du .= w_out * (@. exp(w_in_x + w_b))
end

crnn! (generic function with 1 method)

In [110]:
tspan = [0.0, 1.0];
u0 = zeros(ns);
u0[1] = 1.0;
prob = ODEProblem(crnn!, u0, tspan, p, abstol = lb)

ODEProblem with uType Vector{Float64} and tType Float64. In-place: true
timespan: (0.0, 1.0)
u0: 5-element Vector{Float64}:
 1.0
 0.0
 0.0
 0.0
 0.0

In [111]:
function pred_n_ode(p, i_exp, exp_data)
    #global T0, beta, ocen = l_exp_info[i_exp, :]
    global T0, beta = l_exp_info[i_exp, :]
    global w_in, w_b, w_out = p2vec(p)
    global t0_exp = exp_data[1, 1]
    
    ts = @view(exp_data[:, 1])
    tspan = [ts[1], ts[end]]
    sol = solve(
        prob,
        alg,
        tspan = tspan,
        p = p,
        saveat = ts, 
        sensealg = sense,
        maxiters = maxiters
        # callback = _cb,
    )

    if sol.retcode == :Success
        nothing
    else
        #@sprintf("solver failed beta: %.0f ocen: %.2f", beta, exp(ocen))
        @sprintf("solver failed beta: %.0f", beta)
    end
    if length(sol.t) > length(ts)
        # @show exp_data[:, 1]
        # @show sol.t
        return  sol[:, 1:length(ts)]
    else
        return sol
    end
end

pred_n_ode (generic function with 1 method)

In [112]:
condition(u, t, integrator) = u[1] < lb * 10.0
affect!(integrator) = terminate!(integrator)
_cb = DiscreteCallback(condition, affect!)

#alg = AutoTsit5(TRBDF2(autodiff = true));
alg = TRBDF2(autodiff = true);
#sense = BacksolveAdjoint(checkpointing=true; autojacvec=ZygoteVJP());
sense = ForwardSensitivity(autojacvec = true)

ForwardSensitivity{0, true, Val{:central}}(true, false)

In [113]:
function loss_neuralode(p, i_exp)
    exp_data = l_exp_data[i_exp]
    pred = Array(pred_n_ode(p, i_exp, exp_data))
    masslist = sum(clamp.(@view(pred[1:end-1, :]), 0, Inf), dims = 1)'
    gaslist = clamp.(@views(pred[end, :]), 0, Inf)

    loss = mae(masslist, @view(exp_data[1:length(masslist), 3])) + mae(gaslist, 1 .- @view(exp_data[1:length(masslist), 3]))
    return loss
end

loss_neuralode (generic function with 1 method)

In [114]:
@time loss = loss_neuralode(p,1)

 22.745900 seconds (30.63 M allocations: 1.703 GiB, 11.69% gc time, 99.35% compilation time: 61% of which was recompilation)


1.212574300114602

In [115]:
function plot_sol(i_exp, sol, exp_data, Tlist, cap, sol0 = nothing)
    #T0, beta, ocen = l_exp_info[i_exp, :]
    T0, beta = l_exp_info[i_exp, :]
    
    t0_raw = sol.t[1]                          # actual start time in seconds
    ts     = (sol.t .- t0_raw) ./ 60.0        # ✅ always starts at 0 [min]
    ts_exp = (exp_data[:, 1] .- t0_raw) ./ 60.0
    
    Tlist = [getsampletemp(t, T0, beta, t0_raw) for t in sol.t]
    ind = length(ts)
    plt = plot(
        exp_data[:, 2],
        exp_data[:, 3],
        seriestype = :scatter,
        label = "Exp",
    )

    plot!(
        plt,
        Tlist,
        sum(clamp.(sol[1:end-1, :], 0, Inf), dims = 1)',
        lw = 4,
        legend = :left,
        label = "CRNN",
    )

    if sol0 !== nothing
        plot!(
            plt,
            Tlist0 = [getsampletemp(t, T0, beta) for t in sol0.t],
            sum(sol0[1:end-1, :], dims = 1)',
            label = "initial model",
        )
    end
    xlabel!(plt, "Temperature [K]")
    ylabel!(plt, "Normalized Mass")
    title!(plt, cap)
    exp_cond = string(
        @sprintf(
            # "5s4r \n beta = %.2f K/min",
            "T0 = %.1f K \n beta = %.2f K/min ",
            T0,
            beta,
            #exp(ocen) * 100.0
        )
    )
    annotate!(plt, Tlist[end] * 0.85, 0.4, exp_cond)

    # plt2 = twinx()
    # plot!(
    #     plt2,
    #     exp_data[1:ind, 1] / 60,
    #     Tlist,
    #     lw = 4,
    #     ls = :dash,
    #     legend = :topright,
    #     label = "T",
    # )
    # ylabel!(plt2, "Temp")

    p2 = plot(Tlist, sol[1, :], lw = 4, legend = :right, label = "Xylan")
    for i = 2:ns-1
        plot!(p2, Tlist, sol[i, :], lw = 4, label = "S$i")
    end
    plot!(p2, Tlist, sol[ns, :], lw = 4, label = "Volatile")
    xlabel!(p2, "Temperature [K]")
    ylabel!(p2, "Mass")

    p3 = plot(ts, sol[1, :], lw = 4, legend = :right, label = "Xylan")
    for i = 2:ns-1
        plot!(p3, ts, sol[i, :], lw = 4, label = "S$i")
    end
    plot!(p3, ts, sol[ns, :], lw = 4, label = "Volatile")
    xlabel!(p3, "Time [min]")
    ylabel!(p3, "Mass")

    plt = plot(plt, p2, p3, framestyle = :box, layout = @layout [a; b; c])
    plot!(plt, size = (800, 800))
    return plt
end

cbi = function (p, i_exp)
    exp_data = l_exp_data[i_exp]
    sol = pred_n_ode(p, i_exp, exp_data)
    Tlist = similar(sol.t)
    #T0, beta, ocen = l_exp_info[i_exp, :]
    T0, beta = l_exp_info[i_exp, :]
    for (i, t) in enumerate(sol.t)
        Tlist[i] = getsampletemp(t, T0, beta)
    end
    value = l_exp[i_exp]
    plt = plot_sol(i_exp, sol, exp_data, Tlist, "exp_$value")
    png(plt, string(fig_path, "/conditions/pred_exp_$value"))
    return false
end


function plot_loss(l_loss_train, l_loss_val; yscale = :log10)
    plt_loss = plot(l_loss_train, yscale = yscale, label = "train")
    plot!(plt_loss, l_loss_val, yscale = yscale, label = "val")
    plt_grad = plot(list_grad, yscale = yscale, label = "grad_norm")
    xlabel!(plt_loss, "Epoch")
    ylabel!(plt_loss, "Loss")
    xlabel!(plt_grad, "Epoch")
    ylabel!(plt_grad, "Gradient Norm")
    # ylims!(plt_loss, (-Inf, 1e0))
    # ylims!(plt_grad, (-Inf, 1e3))
    plt_all = plot([plt_loss, plt_grad]..., legend = :top, framestyle=:box)
    plot!(
        plt_all,
        size=(1000, 450),
        xtickfontsize = 11,
        ytickfontsize = 11,
        xguidefontsize = 12,
        yguidefontsize = 12,
    )
    png(plt_all, string(fig_path, "/loss_grad"))
end

l_loss_train = []
l_loss_val = []
list_grad = []
iter = 1
cb = function (p, loss_train, loss_val, g_norm, mlf, exprun)
    global l_loss_train, l_loss_val, list_grad, iter
    push!(l_loss_train, loss_train)
    push!(l_loss_val, loss_val)
    push!(list_grad, g_norm)

    println("LOGGING metrics: iter = ", iter,
        " train = ", loss_train,
        " val = ", loss_val,
        " grad = ", g_norm)
    # Log metrics to MLFlow
    logmetric(mlf, exprun, "train_loss", loss_train, step=iter)
    logmetric(mlf, exprun, "val_loss", loss_val, step=iter)
    logmetric(mlf, exprun, "grad_norm", g_norm, step=iter)
    
    if iter % n_plot == 0
        display_p(p)
        list_exp = randperm(n_exp)[1]
        @printf(
            "Min Loss train: %.2e val: %.2e",
            minimum(l_loss_train),
            minimum(l_loss_val)
        )
        println("\n update plot ", l_exp[list_exp], "\n")
        for i_exp in list_exp
            cbi(p, i_exp)
        end

        plot_loss(l_loss_train, l_loss_val; yscale = :log10)

        @save string(ckpt_path, "/mymodel.bson") p opt l_loss_train l_loss_val list_grad iter
    end
    iter += 1 
end

if is_restart
    @load string(ckpt_path, "/mymodel.bson") p opt l_loss_train l_loss_val list_grad iter
    iter += 1
end

In [116]:
# Log hyperparameters to MLFlow (once, before training loop)
logparam(mlf, exprun, "ns", string(ns))
logparam(mlf, exprun, "nr", string(nr))

logparam(mlf, exprun, "lb", string(lb))
logparam(mlf, exprun, "n_epoch", string(n_epoch))
logparam(mlf, exprun, "n_plot", string(n_plot))
logparam(mlf, exprun, "grad_max", string(grad_max))
logparam(mlf, exprun, "maxiters", string(maxiters))

logparam(mlf, exprun, "lr_max", string(lr_max))
logparam(mlf, exprun, "lr_min", string(lr_min))
logparam(mlf, exprun, "lr_decay", string(lr_decay))
logparam(mlf, exprun, "lr_decay_step", string(lr_decay_step))
logparam(mlf, exprun, "w_decay", string(w_decay))


true

In [117]:
epochs = ProgressBar(iter:n_epoch);
loss_epoch = zeros(Float64, n_exp);
grad_norm = zeros(Float64, n_exp);
for epoch in epochs
    global p
    for i_exp in randperm(n_exp)
        if i_exp in l_val
            continue
        end
        grad = ForwardDiff.gradient(x -> loss_neuralode(x, i_exp), p)
        grad_norm[i_exp] = norm(grad, 2)
        if grad_norm[i_exp] > grad_max
            grad = grad ./ grad_norm[i_exp] .* grad_max
        end
        update!(opt, p, grad)
    end
    for i_exp = 1:n_exp
        loss_epoch[i_exp] = loss_neuralode(p, i_exp) #core of automatic differentiation
    end
    loss_train = mean(loss_epoch[l_train])
    loss_val = mean(loss_epoch[l_val])
    grad_mean = mean(grad_norm[l_train])
    set_description(
        epochs,
        string(
            @sprintf(
                "Loss train: %.2e val: %.2e grad: %.2e lr: %.1e",
                loss_train,
                loss_val,
                grad_mean,
                opt[1].eta
            )
        ),
    )
    cb(p, loss_train, loss_val, grad_mean, mlf, exprun)
end

# conf["loss_train"] = minimum(l_loss_train)
# conf["loss_val"] = minimum(l_loss_val)
#YAML.write_file(config_path, conf)

for i_exp in randperm(n_exp)
    cbi(p, i_exp)
end

0.0%┣                                            ┫ 0/1.5k [00:00<00:-16, -0s/it]


LOGGING metrics: iter = 1 train = 1.0435904859326537 val = 1.0626417355759339 grad = 3.741352667832919


Loss train: 1.04e+00 val: 1.06e+00 grad: 3.74e+00 lr: 5.0e-03 0.1%┣┫ 1/1.5k [01:05<Inf:Inf, InfGs/it]


LOGGING metrics: iter = 2 train = 0.963213563956794 val = 0.9829310398406966 grad = 5.479367103770795


Loss train: 9.63e-01 val: 9.83e-01 grad: 5.48e+00 lr: 5.0e-03 0.1%┣┫ 2/1.5k [01:06<27:16:28, 66s/it]


...

Loss train: 7.07e-02 val: 9.19e-02 grad: 3.07e+00 lr: 4.0e-05 99.9%┣┫ 1.5k/1.5k [34:31<00:03, 1s/it]


LOGGING metrics: iter = 1499 train = 0.07126084822365233 val = 0.09255904392832108 grad = 3.222236710132511


Loss train: 7.13e-02 val: 9.26e-02 grad: 3.22e+00 lr: 4.0e-05 99.9%┣┫ 1.5k/1.5k [34:32<00:01, 1s/it]


LOGGING metrics: iter = 1500 train = 0.07072650914745533 val = 0.09200548919365105 grad = 3.414297232611008

 species (column) reaction (row)
w_in | Ea | b | lnA | w_out
5×13 Matrix{Float64}:
 -0.0    0.0    0.0    3.12   0.0  219.024  0.515  39.42    0.0     0.394   0.229  -3.12   2.497
  0.336  0.021  0.03   0.0    0.0  271.364  0.135  26.417  -0.336  -0.021  -0.03    0.232  0.155
 -0.0    0.0    0.0    0.346  0.0  258.249  0.081  24.966   0.0     0.015   0.054  -0.346  0.277
  1.21   0.0    0.022  0.0    0.0  181.264  0.419  35.976  -1.21    0.021  -0.022   0.887  0.324
 -0.0    0.666  0.352  0.0    0.0  212.945  0.252  30.777   0.0    -0.666  -0.352   0.419  0.599

Min Loss train: 7.07e-02 val: 9.17e-02
 update plot 8



Loss train: 7.07e-02 val: 9.20e-02 grad: 3.41e+00 lr: 1.0e-05 100.0%┣┫ 1.5k/1.5k [34:34<00:00, 1s/it]
Loss train: 7.07e-02 val: 9.20e-02 grad: 3.41e+00 lr: 1.0e-05 100.0%┣┫ 1.5k/1.5k [34:34<00:00, 1s/it]


In [118]:
expr_name = "Xylan-5s5r-01"
fig_path = string("./results_nocat/", expr_name, "/figs")
ckpt_path = string("./results_nocat/", expr_name, "/checkpoint")
@load string(ckpt_path, "/mymodel.bson") p opt l_loss_train l_loss_val list_grad iter

p_cutoff = 0.0

display_p(p)

for i_exp = 1:n_exp
    cbi(p, i_exp)
end

loss_epoch = zeros(Float64, n_exp);
for i_exp = 1:n_exp
    loss_epoch[i_exp] = loss_neuralode(p, i_exp)
end

loss_train = mean(loss_epoch[l_train])
loss_val = mean(loss_epoch[l_val])
loss_train, loss_val


 species (column) reaction (row)
w_in | Ea | b | lnA | w_out
5×13 Matrix{Float64}:
 -0.0    0.0    0.0    3.12   0.0  219.024  0.515  39.42    0.0     0.394   0.229  -3.12   2.497
  0.336  0.021  0.03   0.0    0.0  271.364  0.135  26.417  -0.336  -0.021  -0.03    0.232  0.155
 -0.0    0.0    0.0    0.346  0.0  258.249  0.081  24.966   0.0     0.015   0.054  -0.346  0.277
  1.21   0.0    0.022  0.0    0.0  181.264  0.419  35.976  -1.21    0.021  -0.022   0.887  0.324
 -0.0    0.666  0.352  0.0    0.0  212.945  0.252  30.777   0.0    -0.666  -0.352   0.419  0.599



(0.07072650914745533, 0.09200548919365105)